# Cifar10 classification tricks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deep-learning-ids/deep_learning_fs26/blob/main/notebooks/03b_cifar10_tricks_keras_torch.ipynb)

In this notebook you will download the cifar10 dataset which contains quite small images (32x32x3) of 10 classes. The data is from the Canadian Institute For Advanced Research. You will plot examples of the images with their class labels. Note that because the images are small it is not always very easy to recognise which of the ten classes is on the image, even as a human. After loading the dataset you will train multiple models and compare the performances of the models on the testset.

**Dataset:**  You work with the Cifar10 dataset. You have 60'000 32x32 pixel color images of 10 classes ("airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck")

**Content:**
* load the original cifar10 data create a train val and test dataset
* visualize samples of cifar10 dataset

* train a random forest on the pixelvalues
* train a cnn from scratch
* train a cnn with dropout to correct fro overfitting
* train a cnn with data augmentation
* train a cnn with data augmentation and more filters (wider)
* train a cnn with data augmentation and more layers (deeper)
* train a cnn with data augmentation, wider and deeper.
* add batch normalization
* add weight regularization

* compare the performances of the models


* HOW TO IMPROVE SYSTEMATICALLY? Error analysis



* 🔧  your task at the end of the notebook: try to beat the models with your own improvement ideas.


# Helping functions

In [ ]:
# -----------------------------
# Helping function: history plotting
# -----------------------------
import numpy as np
import matplotlib.pyplot as plt

def plot_history(history):
  # plot the development of the accuracy and loss during training
  plt.figure(figsize=(12,4))
  plt.subplot(1,2,(1))
  plt.plot(history.history['accuracy'], linestyle='-.')
  plt.plot(history.history['val_accuracy'])
  plt.title('model accuracy')
  plt.ylabel('accuracy')
  plt.xlabel('epoch')
  plt.legend(['train', 'valid'], loc='lower right')

  plt.subplot(1,2,(2))
  plt.plot(history.history['loss'], linestyle='-.')
  plt.plot(history.history['val_loss'])
  plt.title('model loss')
  plt.ylabel('loss')
  plt.xlabel('epoch')
  plt.legend(['train', 'valid'], loc='upper right')
  plt.show()

In [ ]:
# check if we are using GPU
import torch
torch.cuda.is_available()

# Data loading

In the next cell you will load the Cifar10 dataset, 50'000 images are in the training set and 10'000 are in the test dataset. You will use 10'000 for the train and validation dataset. We will plot one random example of each label.

In [ ]:
# -----------------------------
# Load CIFAR-10
# -----------------------------
from keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# separate train val and test dataset
X_train = x_train[0:10000]
Y_train = to_categorical(y_train[0:10000], 10)  # one-hot encoding

X_val = x_train[20000:30000]
Y_val = to_categorical(y_train[20000:30000], 10)

X_test = x_test
Y_test = to_categorical(y_test, 10)

del x_train, y_train, x_test, y_test

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

# sample image of each label
labels = np.array(["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"])
plt.figure(figsize=(15,15))
for i in range(0, len(np.unique(np.argmax(Y_train, axis=1)))):
    rmd = np.random.choice(np.where(np.argmax(Y_train, axis=1) == i)[0], 1)
    plt.subplot(1,10,i+1)
    img = X_train[rmd]
    plt.imshow(img[0,:,:,:])
    plt.title(labels[i] + " " + str(np.argmax(Y_train, axis=1)[rmd][0]))
plt.show()

# check the shape of the data
print(X_train.shape, Y_train.shape, X_val.shape, Y_val.shape)

# Random forest

In [ ]:
# ============================================================
# Random Forest baseline on raw pixels
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Labels: convert one-hot to class indices (RF needs 1D integer labels)
y_train_idx = np.argmax(Y_train, axis=1).astype(np.int64)
y_val_idx   = np.argmax(Y_val, axis=1).astype(np.int64)
y_test_idx  = np.argmax(Y_test, axis=1).astype(np.int64)

# Flatten images to vectors: (N, 32, 32, 3) -> (N, 3072)
X_train_rf = X_train.reshape(len(X_train), -1).astype(np.uint8)
X_val_rf   = X_val.reshape(len(X_val), -1).astype(np.uint8)
X_test_rf  = X_test.reshape(len(X_test), -1).astype(np.uint8)

# Random Forest (keep it modest; RF on full 10k x 3072 can be heavy)
rf = RandomForestClassifier(
    n_estimators=40,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
    class_weight=None
)

rf.fit(X_train_rf, y_train_idx)

# Evaluate on test set
rf_test_pred = rf.predict(X_test_rf)
rf_test_acc = accuracy_score(y_test_idx, rf_test_pred)
print(f"Random Forest test accuracy: {rf_test_acc:.4f}")

# Optional: quick report (comment out if too verbose)
# print(classification_report(y_test_idx, rf_test_pred))

# Store results (so you can concat later with CNN results)
res_rf = pd.DataFrame({'Acc': [rf_test_acc]}, index=['Random Forest (raw pixels)'])
res_rf


Think: what accuracy do you expect with no learning at all?

# Baseline CNN model

Baseline architecture considerations:
1) CIFAR-10 images are tiny (32×32). You don’t need a very deep network to start extracting useful patterns. keep small to train fast (not SOTA):
- Stage 1 (16 filters) learns low-level features: edges, corners, simple color/texture patterns.
- Stage 2 (32 filters) learns more complex combinations: parts of objects, textures, simple shapes.
2) Small 3×3 kernels are the standard building block (increases effective receptive field with fewer parameters and more nonlinearity).
3) Pooling reduces resolution and adds translation tolerance: MaxPooling2D((2,2)) halves spatial resolution:
- 32×32 → 16×16 after first pool
- 16×16 → 8×8 after second pool

4) Modern CNNs follow this rule almost universally: As spatial resolution decreases, channel depth increases:

- At 32×32: You don’t need many filters to detect simple edges. 16 filters is enough for basic low-level features.

- At 8×8: Each unit corresponds to a much larger part of the image. The network now needs more filters to encode different object parts and class-specific combinations. 32 filters (or more in larger networks) make sense here.

5) A small dense head keeps the model interpretable and limits memorization (overfitting).

In [ ]:
# -----------------------------
# CNN
# -----------------------------
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation

model = Sequential()

model.add(Conv2D(16, (3,3), activation="relu", padding="same", input_shape=(32,32,3)))
model.add(Conv2D(16, (3,3), activation="relu", padding="same"))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(32, (3,3), activation="relu", padding="same"))
model.add(Conv2D(32, (3,3), activation="relu", padding="same"))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())
model.add(Dense(128))
model.add(Activation('relu'))
model.add(Dense(10))
model.add(Activation('softmax'))

# use loss='sparse_categorical_crossentropy' when labels are integers instead of one-hot encoded vectors
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:


# ============================================================
# Data and label preparation
# ============================================================

# 1) Convert images to float32 and scale to [0,1]
X_train_tf = X_train.astype("float32") / 255.0
X_val_tf   = X_val.astype("float32") / 255.0
X_test_tf  = X_test.astype("float32") / 255.0

print("Train min/max:", X_train_tf.min(), X_train_tf.max())

# 2) Convert one-hot labels to integer indices (required for sparse_categorical_crossentropy)
y_train_idx = np.argmax(Y_train, axis=1).astype("int64")
y_val_idx   = np.argmax(Y_val, axis=1).astype("int64")
y_test_idx  = np.argmax(Y_test, axis=1).astype("int64")

print("Label example:", y_train_idx[:10])

In [ ]:


# -----------------------------
# Train (fit) + evaluate
# -----------------------------
EPOCHS = 15
BATCH_SIZE = 64

history = model.fit(
    X_train_tf, y_train_idx,
    validation_data=(X_val_tf, y_val_idx),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2
)

plot_history(history)

# Keras evaluation
cnn_test_loss, cnn_test_acc = model.evaluate(X_test_tf, y_test_idx, batch_size=BATCH_SIZE, verbose=0)
print(f"Test accuracy (model.evaluate): {cnn_test_acc:.4f}")

# manual accuracy (also possible)
probs = model.predict(X_test_tf, batch_size=BATCH_SIZE, verbose=0)
acc_manual = np.mean(np.argmax(probs, axis=1) == y_test_idx)
print(f"Test accuracy (manual):        {acc_manual:.4f}")


In [ ]:
# after CNN evaluation gives cnn_test_acc
res_cnn = pd.DataFrame({'Acc': [cnn_test_acc]}, index=['CNN (from scratch)'])
pd.concat([res_rf, res_cnn])

# Dropout

Val_acc << train_acc means that our baseline model overfits (memorizes the training set). Dropout regulizes the model to prevent overfitting.

Note: we apply dropout on the fully connected layers and not the conv layers. why?
- Convolutional layers learn local spatial patterns (edges, textures, shapes).

- Standard Dropout randomly zeros individual activations.

- That breaks spatial continuity inside feature maps.

- It can damage low-level feature learning more than it helps regularization.

- In contrast: dense layers do not rely on spatial structure.

- Dropout works very naturally there by preventing co-adaptation of neurons.

- So in classical CNN design: Dropout → mostly in dense layers

- Conv layers → use BatchNorm or weight decay instead (see below)).

In [ ]:
from tensorflow.keras.layers import Dropout
model_do = Sequential()

model_do.add(Conv2D(16, (3,3), activation="relu", padding="same", input_shape=(32,32,3)))
model_do.add(Conv2D(16, (3,3), activation="relu", padding="same"))
model_do.add(MaxPooling2D((2,2)))

model_do.add(Conv2D(32, (3,3), activation="relu", padding="same"))
model_do.add(Conv2D(32, (3,3), activation="relu", padding="same"))
model_do.add(MaxPooling2D((2,2)))

model_do.add(Flatten())
model_do.add(Dense(128))
model_do.add(Activation('relu'))
model_do.add(Dropout(0.5))   # <- only dropout


model_do.add(Dense(10))
model_do.add(Activation('softmax'))


model_do.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_do.summary()

In [ ]:
history_do_raw = model_do.fit(
    X_train_tf, y_train_idx,
    validation_data=(X_val_tf, y_val_idx),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2
)

do_raw_test_loss, do_raw_test_acc = model_do.evaluate(X_test_tf, y_test_idx, batch_size=BATCH_SIZE, verbose=0)
print(f"Test accuracy (model_do.evaluate): {do_raw_test_acc:.4f}")

plot_history(history_do_raw)

res_do_raw = pd.DataFrame({"Acc":[do_raw_test_acc]},index=["CNN  + dropout"])
res_do_raw

# ------------------------------------------------------------
# Compare to baseline CNN results

# ------------------------------------------------------------
pd.concat([res_rf,res_cnn, res_do_raw])

## Effect of dropout

Training accuracy drops, test accuracy slightly increases. What does this imply? dropout largely removed overfitting (still some left, can add more dropout layers). No big accuracy improvement on test data means that the main classification difficulty is not due to overfitting on the training data but rather the quality of the data and /or the prediction capacity of the CNN model. In the following we will improve generalization by
1) Augmenting the data.
2) increasing the model capacity.

# Data augmentation
1. RandomFlip("horizontal"): randomly mirrors the image left↔right to make the model invariant to horizontal orientation changes.

2. ZeroPadding2D(4): adds a 4-pixel border of zeros around the image so later random crops can shift the content without losing size.

3. RandomCrop(32, 32): takes a random 32×32 window from a larger (padded) image to simulate small translations and framing changes.

The pipeline ds_train_aug does:

- Create dataset from arrays

- Shuffle samples

- Group into batches

- Apply augmentation to each batch

- Preload the next batch for performance

In [ ]:
# ------------------------------------------------------------
# Data augmentation pipeline (applied only to training data)
# ------------------------------------------------------------
batch_size = 64

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.ZeroPadding2D(4),
    tf.keras.layers.RandomCrop(32, 32),
], name="augment")


ds_train_aug = tf.data.Dataset.from_tensor_slices((X_train_tf, y_train_idx)).shuffle(10000).batch(batch_size).map(lambda x,y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
ds_val = tf.data.Dataset.from_tensor_slices((X_val_tf, y_val_idx)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
ds_test = tf.data.Dataset.from_tensor_slices((X_test_tf, y_test_idx)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

## Visualize the augmentations

In [ ]:

import matplotlib.pyplot as plt

# Pick one example image (assumes X_train_tf exists in [0,1] and shape (N,32,32,3))
idx = np.random.randint(0, X_train_tf.shape[0])
img = X_train_tf[idx]  # (32,32,3), float32

# Define individual augmentation layers
flip_layer = tf.keras.layers.RandomFlip("horizontal")
pad_layer  = tf.keras.layers.ZeroPadding2D(4)
crop_layer = tf.keras.layers.RandomCrop(32, 32)

# Apply augmentations (need a batch dimension)
img_b = tf.expand_dims(img, axis=0)  # (1,32,32,3)

img_flip = flip_layer(img_b, training=True)[0]
img_pad  = pad_layer(img_b, training=True)[0]            # (40,40,3)

# RandomCrop expects input >= (32,32), so crop from padded image
img_crop = crop_layer(tf.expand_dims(img_pad, axis=0), training=True)[0]  # back to (32,32,3)

# Plot
plt.figure(figsize=(14,4))

plt.subplot(1,4,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(img_flip)
plt.title("RandomFlip (horizontal)")
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(img_pad)
plt.title("ZeroPadding2D(4) → 40×40")
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(img_crop)
plt.title("Pad + RandomCrop(32×32)")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Example index:", idx)


In [ ]:
# ------------------------------------------------------------
# baseline CNN again
# ------------------------------------------------------------
model_aug = Sequential()
model_aug.add(Conv2D(16,(3,3),activation="relu",padding="same",input_shape=(32,32,3)))
model_aug.add(Conv2D(16,(3,3),activation="relu",padding="same"))
model_aug.add(MaxPooling2D((2,2)))
model_aug.add(Conv2D(32,(3,3),activation="relu",padding="same"))
model_aug.add(Conv2D(32,(3,3),activation="relu",padding="same"))
model_aug.add(MaxPooling2D((2,2)))
model_aug.add(Flatten())
model_aug.add(Dense(128))
model_aug.add(Activation('relu'))
model_aug.add(Dense(10))
model_aug.add(Activation('softmax'))
model_aug.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy"])
model_aug.summary()



In [ ]:
# ------------------------------------------------------------
# Train on augmented data, validate on the same val set
# ------------------------------------------------------------
epochs = 15
history_aug = model_aug.fit(ds_train_aug, validation_data=ds_val, epochs=epochs, verbose=2)

plot_history(history_aug)

# ------------------------------------------------------------
# Test on the same test data
# ------------------------------------------------------------
aug_test_loss, aug_test_acc = model_aug.evaluate(ds_test, verbose=0)
print("Augmented training model test accuracy:", aug_test_acc)

# ------------------------------------------------------------
# Compare to baseline CNN results
# res_cnn should exist from your earlier baseline CNN run
# If res_cnn doesn't exist, create it from your stored baseline test accuracy variable
# ------------------------------------------------------------
res_aug = pd.DataFrame({"Acc":[aug_test_acc]}, index=["CNN + augmentation"])
pd.concat([res_rf,res_cnn, res_do_raw,res_aug])


## Effect of augmentation

augmentation makes the training problem harder:
The model sees shifted / flipped / cropped images --> Can no longer memorize exact pixel positions

--> Training accuracy further decreases, this is expected and healthy.

--> Data augmentation helps generalize: no overfitting and thus higher test accuracy. Further improvements could be possibly achieved if we increase the model's capability with more conv layers.

# Increase CNN capacity

There are 2 main ways to increase capacity in a CNN:

1. Go deeper (add more Conv layers)

2. Go wider (increase number of filters per layer)

Both increase representational power, but they behave differently. Increasing the number of feature maps we

- let each spatial level learn more patterns

- Increase modeling capacity without changing gradient depth --> keep optimization stable

In [ ]:
model_aug_bigconv = Sequential()
model_aug_bigconv.add(Conv2D(32,(3,3),activation="relu",padding="same",input_shape=(32,32,3))) # x2 more filters
model_aug_bigconv.add(Conv2D(32,(3,3),activation="relu",padding="same"))                       # x2 more filters
model_aug_bigconv.add(MaxPooling2D((2,2)))
model_aug_bigconv.add(Conv2D(64,(3,3),activation="relu",padding="same"))                       # x2 more filters
model_aug_bigconv.add(Conv2D(64,(3,3),activation="relu",padding="same"))                       # x2 more filters
model_aug_bigconv.add(MaxPooling2D((2,2)))
model_aug_bigconv.add(Flatten())
model_aug_bigconv.add(Dense(128))
model_aug_bigconv.add(Activation("relu"))
model_aug_bigconv.add(Dense(10))
model_aug_bigconv.add(Activation("softmax"))
model_aug_bigconv.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy"])
model_aug_bigconv.summary()

history_aug_bigconv = model_aug_bigconv.fit(ds_train_aug,validation_data=ds_val,epochs=epochs,verbose=2)
plot_history(history_aug_bigconv)

bigconv_test_loss,bigconv_test_acc = model_aug_bigconv.evaluate(ds_test,verbose=0)
print("CNN (wider) + augmentation test accuracy:",bigconv_test_acc)

res_aug_bigconv = pd.DataFrame({"Acc":[bigconv_test_acc]},index=["CNN (wider) + augmentation"])
pd.concat([res_rf,res_cnn, res_do_raw,res_aug,res_aug_bigconv])


# Going deeper

In [ ]:


model_deeper3 = Sequential()
model_deeper3.add(Conv2D(16,(3,3),activation="relu",padding="same",input_shape=(32,32,3)))
model_deeper3.add(Conv2D(16,(3,3),activation="relu",padding="same"))
model_deeper3.add(MaxPooling2D((2,2)))
model_deeper3.add(Conv2D(32,(3,3),activation="relu",padding="same"))
model_deeper3.add(Conv2D(32,(3,3),activation="relu",padding="same"))
model_deeper3.add(MaxPooling2D((2,2)))
model_deeper3.add(Conv2D(64,(3,3),activation="relu",padding="same"))
model_deeper3.add(Conv2D(64,(3,3),activation="relu",padding="same"))
model_deeper3.add(MaxPooling2D((2,2)))
model_deeper3.add(Flatten())
model_deeper3.add(Dense(128))
model_deeper3.add(Activation("relu"))
model_deeper3.add(Dense(10))
model_deeper3.add(Activation("softmax"))
model_deeper3.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy"])
model_deeper3.summary()

epochs = 30 # train longer
history_deeper3 = model_deeper3.fit(ds_train_aug,validation_data=ds_val,epochs=epochs,verbose=2)
plot_history(history_deeper3)

deeper3_test_loss,deeper3_test_acc = model_deeper3.evaluate(ds_test,verbose=0)
print("Deeper CNN  + augmentation test accuracy:",deeper3_test_acc)

res_deeper3 = pd.DataFrame({"Acc":[deeper3_test_acc]},index=["CNN (deeper) + augmentation"])


# ------------------------------------------------------------
# Compare to baseline CNN results

# ------------------------------------------------------------
pd.concat([res_rf,res_cnn, res_do_raw,res_aug,res_aug_bigconv,res_deeper3])


# Both wider and deeper

Note : now we must also train longer.


aim: get to overfitting to show the effect of dropout /BN

In [ ]:
model_deeper3 = Sequential()
model_deeper3.add(Conv2D(32,(3,3),activation="relu",padding="same",input_shape=(32,32,3)))
model_deeper3.add(Conv2D(32,(3,3),activation="relu",padding="same"))
model_deeper3.add(MaxPooling2D((2,2)))
model_deeper3.add(Conv2D(64,(3,3),activation="relu",padding="same"))
model_deeper3.add(Conv2D(64,(3,3),activation="relu",padding="same"))
model_deeper3.add(MaxPooling2D((2,2)))
model_deeper3.add(Conv2D(128,(3,3),activation="relu",padding="same"))
model_deeper3.add(Conv2D(128,(3,3),activation="relu",padding="same"))
model_deeper3.add(MaxPooling2D((2,2)))
model_deeper3.add(Flatten())
model_deeper3.add(Dense(128))
model_deeper3.add(Activation("relu"))
model_deeper3.add(Dense(10))
model_deeper3.add(Activation("softmax"))
model_deeper3.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy"])
model_deeper3.summary()


epochs = 30 # train longer

history_deeper3 = model_deeper3.fit(ds_train_aug,validation_data=ds_val,epochs=epochs,verbose=2)
plot_history(history_deeper3)

deeper3_test_loss,deeper3_test_acc = model_deeper3.evaluate(ds_test,verbose=0)
print("Deeper and wider CNN + augmentation test accuracy:",deeper3_test_acc)

res_bigger = pd.DataFrame({"Acc":[deeper3_test_acc]},index=["CNN deeper & wider + augmentation"])
res_bigger

# ------------------------------------------------------------
# Compare to baseline CNN results

# ------------------------------------------------------------
pd.concat([res_rf,res_cnn, res_do_raw,res_aug,res_aug_bigconv,res_deeper3,res_bigger])



1️⃣ Large head, no augmentation

Train >> Val → overfitting → regularization helps

2️⃣ Augmentation

Train ≈ Val → reduced overfitting

3️⃣ Augmentation + deeper trunk

Now capacity limits performance → need better features

# Batch Normalization
try to improve a large model using BN

In [ ]:

from tensorflow.keras.layers import BatchNormalization
# use_bias = False:
#Because BatchNorm already includes a learnable shift (bias) term.
#So the bias in the preceding Conv/Dense layer becomes redundant

model_deeper3_bn = Sequential()

model_deeper3_bn.add(Conv2D(32,(3,3),padding="same",input_shape=(32,32,3),use_bias=False))
model_deeper3_bn.add(BatchNormalization())
model_deeper3_bn.add(Activation("relu"))

model_deeper3_bn.add(Conv2D(32,(3,3),padding="same",use_bias=False))
model_deeper3_bn.add(BatchNormalization())
model_deeper3_bn.add(Activation("relu"))

model_deeper3_bn.add(MaxPooling2D((2,2)))

model_deeper3_bn.add(Conv2D(64,(3,3),padding="same",use_bias=False))
model_deeper3_bn.add(BatchNormalization())
model_deeper3_bn.add(Activation("relu"))

model_deeper3_bn.add(Conv2D(64,(3,3),padding="same",use_bias=False))
model_deeper3_bn.add(BatchNormalization())
model_deeper3_bn.add(Activation("relu"))

model_deeper3_bn.add(MaxPooling2D((2,2)))

model_deeper3_bn.add(Conv2D(128,(3,3),padding="same",use_bias=False))
model_deeper3_bn.add(BatchNormalization())
model_deeper3_bn.add(Activation("relu"))

model_deeper3_bn.add(Conv2D(128,(3,3),padding="same",use_bias=False))
model_deeper3_bn.add(BatchNormalization())
model_deeper3_bn.add(Activation("relu"))

model_deeper3_bn.add(MaxPooling2D((2,2)))

model_deeper3_bn.add(Flatten())

model_deeper3_bn.add(Dense(128,use_bias=False))
model_deeper3_bn.add(BatchNormalization())
model_deeper3_bn.add(Activation("relu"))

model_deeper3_bn.add(Dense(10))
model_deeper3_bn.add(Activation("softmax"))

model_deeper3_bn.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model_deeper3_bn.summary()


epochs = 45 # train longer
history_deeper3_bn = model_deeper3_bn.fit(
    ds_train_aug,
    validation_data=ds_val,
    epochs=epochs,
    verbose=2
)

plot_history(history_deeper3_bn)

deeper3_bn_test_loss, deeper3_bn_test_acc = model_deeper3_bn.evaluate(ds_test, verbose=0)
print("Deeper & wider CNN  + BN + augmentation test accuracy:", deeper3_bn_test_acc)

res_deeper3_bn = pd.DataFrame(
    {"Acc":[deeper3_bn_test_acc]},
    index=["CNN deeper & wider + BN + augmentation"]
)

pd.concat([res_cnn, res_aug, res_aug_bigconv, res_deeper3, res_bigger, res_deeper3_bn])


In [ ]:
## Effect of BN

## Effect of BN
With BN and longer training we increase training and testing accuracy.
BN improves optimization dynamics, not generalization strength.

--> Increase train accuracy faster than test accuracy.
--> Introduce regularization in addition using weight decay.


# Weight decay

In [ ]:
from tensorflow.keras import regularizers

wd = 1e-4  # weight decay (L2). common CIFAR starting point

model_deeper3_bn_wd = Sequential()

model_deeper3_bn_wd.add(Conv2D(32,(3,3),padding="same",input_shape=(32,32,3),use_bias=False,kernel_regularizer=regularizers.l2(wd)))
model_deeper3_bn_wd.add(BatchNormalization())
model_deeper3_bn_wd.add(Activation("relu"))

model_deeper3_bn_wd.add(Conv2D(32,(3,3),padding="same",use_bias=False,kernel_regularizer=regularizers.l2(wd)))
model_deeper3_bn_wd.add(BatchNormalization())
model_deeper3_bn_wd.add(Activation("relu"))

model_deeper3_bn_wd.add(MaxPooling2D((2,2)))

model_deeper3_bn_wd.add(Conv2D(64,(3,3),padding="same",use_bias=False,kernel_regularizer=regularizers.l2(wd)))
model_deeper3_bn_wd.add(BatchNormalization())
model_deeper3_bn_wd.add(Activation("relu"))

model_deeper3_bn_wd.add(Conv2D(64,(3,3),padding="same",use_bias=False,kernel_regularizer=regularizers.l2(wd)))
model_deeper3_bn_wd.add(BatchNormalization())
model_deeper3_bn_wd.add(Activation("relu"))

model_deeper3_bn_wd.add(MaxPooling2D((2,2)))

model_deeper3_bn_wd.add(Conv2D(128,(3,3),padding="same",use_bias=False,kernel_regularizer=regularizers.l2(wd)))
model_deeper3_bn_wd.add(BatchNormalization())
model_deeper3_bn_wd.add(Activation("relu"))

model_deeper3_bn_wd.add(Conv2D(128,(3,3),padding="same",use_bias=False,kernel_regularizer=regularizers.l2(wd)))
model_deeper3_bn_wd.add(BatchNormalization())
model_deeper3_bn_wd.add(Activation("relu"))

model_deeper3_bn_wd.add(MaxPooling2D((2,2)))

model_deeper3_bn_wd.add(Flatten())

model_deeper3_bn_wd.add(Dense(128,use_bias=False,kernel_regularizer=regularizers.l2(wd)))
model_deeper3_bn_wd.add(BatchNormalization())
model_deeper3_bn_wd.add(Activation("relu"))

model_deeper3_bn_wd.add(Dense(10,kernel_regularizer=regularizers.l2(wd)))
model_deeper3_bn_wd.add(Activation("softmax"))

model_deeper3_bn_wd.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model_deeper3_bn_wd.summary()

epochs = 45

history_deeper3_bn_wd = model_deeper3_bn_wd.fit(
    ds_train_aug,
    validation_data=ds_val,
    epochs=epochs,
    verbose=2
)

plot_history(history_deeper3_bn_wd)

deeper3_bn_wd_test_loss, deeper3_bn_wd_test_acc = model_deeper3_bn_wd.evaluate(ds_test, verbose=0)
print("Deeper CNN  + BN + augmentation + weight decay test accuracy:", deeper3_bn_wd_test_acc)

res_deeper3_bn_wd = pd.DataFrame(
    {"Acc":[deeper3_bn_wd_test_acc]},
    index=[f"CNN deeper + BN + aug + wd={wd:g}"]
)

pd.concat([res_cnn, res_aug, res_aug_bigconv, res_deeper3, res_bigger, res_deeper3_bn, res_deeper3_bn_wd])


## Effect of Weight decay
BatchNorm is primarily an optimization tool.
Weight decay, dropout, and augmentation are regularization tools.

# Error analysis

# Confusion matrix

In [ ]:

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Get predictions
y_pred_probs = model_deeper3_bn_wd.predict(ds_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test_idx  # your true labels (not one-hot)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=labels)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()


# Per-class accuracy

In [ ]:
class_acc = cm.diagonal() / cm.sum(axis=1)

for i, acc in enumerate(class_acc):
    print(f"{labels[i]:12s}: {acc:.3f}")


# Visualize misclassified examples

In [ ]:
mis_idx = np.where(y_pred != y_true)[0]

plt.figure(figsize=(12,8))
for i in range(12):
    idx = mis_idx[i]
    plt.subplot(3,4,i+1)
    plt.imshow(X_test_tf[idx])
    plt.title(f"T:{labels[y_true[idx]]}\nP:{labels[y_pred[idx]]}")
    plt.axis("off")
plt.tight_layout()
plt.show()


# High confidence errors

In [ ]:
conf = np.max(y_pred_probs, axis=1)
wrong_conf = conf[mis_idx]

high_conf_errors = mis_idx[np.argsort(-wrong_conf)]

plt.figure(figsize=(12,8))
for i in range(12):
    idx = high_conf_errors[i]
    plt.subplot(3,4,i+1)
    plt.imshow(X_test_tf[idx])
    plt.title(f"T:{labels[y_true[idx]]}\nP:{labels[y_pred[idx]]}\nConf:{conf[idx]:.2f}")
    plt.axis("off")
plt.tight_layout()
plt.show()


# Low confidence correct classifications

In [ ]:
correct_idx = np.where(y_pred == y_true)[0]
low_conf_correct = correct_idx[np.argsort(conf[correct_idx])]

plt.figure(figsize=(12,8))
for i in range(12):
    idx = low_conf_correct[i]
    plt.subplot(3,4,i+1)
    plt.imshow(X_test_tf[idx])
    plt.title(f"T:{labels[y_true[idx]]}\nConf:{conf[idx]:.2f}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## 🔧 **YOUR TASK:**
- Try to beat the performace of the best network with your own neural network.  
- you might want to combine some approaches from above
- dont forget to normalize also the testset if you use normalization (which you should use anyway😉)







In [ ]:
### YOUR CODE ###